# Crypto Technical Indicators Calculator & Visualizer


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Technical Indicators Calculator & Visualizer

This notebook demonstrates how to compute and visualize key technical indicators used in crypto market analysis: SMA, EMA, RSI, and MACD. Based on concepts from crypto chart analysis guides, this tool helps traders understand indicator signals and candlestick patterns in practice.

**Learning Outcomes:**
- Compute moving averages (SMA/EMA)
- Calculate RSI for overbought/oversold conditions
- Generate MACD crossover signals
- Visualize indicators on price charts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

plt.style.use('seaborn-v0_8-darkgrid')

print('Loading BTC/USD historical data...')
start_date = datetime.now() - timedelta(days=180)
btc_data = yf.download('BTC-USD', start=start_date, end=datetime.now(), progress=False)
print(f'Data loaded: {len(btc_data)} candles')
print(btc_data.head())

## Moving Averages (SMA & EMA)

**Simple Moving Average (SMA):** Arithmetic mean of price over N periods. Smooths price noise.

**Exponential Moving Average (EMA):** Weighted average giving more weight to recent prices. Responds faster to price changes.

In [ ]:
def calculate_sma(prices, period):
    return prices.rolling(window=period).mean()

def calculate_ema(prices, period):
    return prices.ewm(span=period, adjust=False).mean()

btc_data['SMA_20'] = calculate_sma(btc_data['Close'], 20)
btc_data['SMA_50'] = calculate_sma(btc_data['Close'], 50)
btc_data['EMA_12'] = calculate_ema(btc_data['Close'], 12)
btc_data['EMA_26'] = calculate_ema(btc_data['Close'], 26)

print('Moving Averages calculated:')
print(btc_data[['Close', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26']].tail(10))

## Relative Strength Index (RSI)

**RSI** measures momentum on a 0–100 scale:
- **70+:** Overbought (potential sell signal)
- **30-:** Oversold (potential buy signal)
- **Formula:** RSI = 100 - (100 / (1 + RS)), where RS = Avg Gain / Avg Loss over N periods

In [ ]:
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

btc_data['RSI_14'] = calculate_rsi(btc_data['Close'], 14)

print('RSI calculated:')
print(btc_data[['Close', 'RSI_14']].tail(10))
print(f'\nCurrent RSI: {btc_data["RSI_14"].iloc[-1]:.2f}')
if btc_data['RSI_14'].iloc[-1] > 70:
    print('Signal: OVERBOUGHT (potential reversal)')
elif btc_data['RSI_14'].iloc[-1] < 30:
    print('Signal: OVERSOLD (potential bounce)')

## MACD (Moving Average Convergence Divergence)

**MACD** identifies trend changes:
- **MACD Line:** EMA(12) - EMA(26)
- **Signal Line:** EMA(9) of MACD
- **Histogram:** MACD - Signal (momentum)
- **Crossovers:** MACD above Signal = bullish; below = bearish

In [ ]:
def calculate_macd(prices, fast=12, slow=26, signal=9):
    ema_fast = calculate_ema(prices, fast)
    ema_slow = calculate_ema(prices, slow)
    macd = ema_fast - ema_slow
    signal_line = calculate_ema(macd, signal)
    histogram = macd - signal_line
    return macd, signal_line, histogram

btc_data['MACD'], btc_data['MACD_Signal'], btc_data['MACD_Histogram'] = calculate_macd(btc_data['Close'])

print('MACD calculated:')
print(btc_data[['Close', 'MACD', 'MACD_Signal', 'MACD_Histogram']].tail(10))
macd_current = btc_data['MACD'].iloc[-1]
signal_current = btc_data['MACD_Signal'].iloc[-1]
print(f'\nCurrent MACD: {macd_current:.2f}')
print(f'Current Signal: {signal_current:.2f}')
if macd_current > signal_current:
    print('Signal: BULLISH (MACD above signal line)')
else:
    print('Signal: BEARISH (MACD below signal line)')

## Visualize All Indicators

Plot candlestick prices with moving averages, RSI, and MACD on separate panels to see how indicators align with price action.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax1, ax2, ax3 = axes

recent_data = btc_data.tail(90)

ax1.plot(recent_data.index, recent_data['Close'], label='Close Price', linewidth=2, color='black')
ax1.plot(recent_data.index, recent_data['SMA_20'], label='SMA 20', alpha=0.7, linestyle='--')
ax1.plot(recent_data.index, recent_data['SMA_50'], label='SMA 50', alpha=0.7, linestyle='--')
ax1.fill_between(recent_data.index, recent_data['Close'].min() * 0.95, recent_data['Close'].max() * 1.05, alpha=0.1)
ax1.set_ylabel('Price (USD)', fontsize=11)
ax1.set_title('BTC/USD Price & Moving Averages', fontsize=12, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

ax2.plot(recent_data.index, recent_data['RSI_14'], label='RSI(14)', color='purple', linewidth=2)
ax2.axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.fill_between(recent_data.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(recent_data.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylabel('RSI', fontsize=11)
ax2.set_ylim([0, 100])
ax2.set_title('Relative Strength Index (RSI)', fontsize=12, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

colors = ['green' if h > 0 else 'red' for h in recent_data['MACD_Histogram']]
ax3.bar(recent_data.index, recent_data['MACD_Histogram'], label='MACD Histogram', color=colors, alpha=0.3)
ax3.plot(recent_data.index, recent_data['MACD'], label='MACD', color='blue', linewidth=2)
ax3.plot(recent_data.index, recent_data['MACD_Signal'], label='Signal Line', color='red', linewidth=2, linestyle='--')
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax3.set_ylabel('MACD', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.set_title('MACD & Signal Line', fontsize=12, fontweight='bold')
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nChart rendered successfully!')

## Summary & Interpretation Guide

**Golden Cross:** SMA(20) crosses above SMA(50) = bullish signal

**Death Cross:** SMA(20) crosses below SMA(50) = bearish signal

**RSI Signals:**
- Above 70 in uptrend = confirmation; in downtrend = divergence (warning)
- Below 30 in downtrend = confirmation; in uptrend = divergence (warning)

**MACD Signals:**
- Histogram turns from red to green = bullish crossover
- Histogram turns from green to red = bearish crossover
- Divergence between price & MACD = potential reversal

Always combine multiple indicators; no single indicator is 100% reliable.

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/crypto-market-analysis-chart)
